# Data Provenance and Endpoint Definitions

## Scientific objective
Freeze source, access date, version, license status, citations, label encodings, units, assay context, Tox21 missingness, and one dataset card per endpoint.

## Inputs
- `data/metadata/dataset_registry.csv`
- Raw source tables

## Expected outputs
- `data/metadata/dataset_cards/*.md`
- `data/metadata/tox21_missingness.csv`
- `reports/provenance_validation.json`

## Dependencies
pandas, PyYAML

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
hERG is binary at the documented 10 µM IC50 threshold. Ames is the distributed aggregate binary target. Tox21 blanks are unavailable labels, not negatives.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
The default Ames table cannot support strain/metabolic-activation stratification; source enrichment is a separate provenance-preserving task.

## Next notebook
[04_molecular_standardization.ipynb](./04_molecular_standardization.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723}


In [2]:
from toxicity_screening.data_download import read_csv_preserve_missing
from toxicity_screening.utils import atomic_write_json
registry = pd.read_csv(ROOT / "data/metadata/dataset_registry.csv")
card_dir = ROOT / "data/metadata/dataset_cards"
card_dir.mkdir(parents=True, exist_ok=True)
for row in registry.itertuples(index=False):
    text = f"""# Dataset card: {row.dataset_name}

- Endpoint: {row.endpoint}
- Source/provider: {row.source}
- Official URL: {row.official_url}
- Version: {row.version}
- Download date: {row.download_date}
- License/terms status: {row.license}
- Citation: {row.citation}
- Raw file: {row.raw_filename}
- SHA-256: `{row.checksum}`
- Raw records: {row.number_of_records}
- Label definition: {row.label_definition}
- Units: {row.measurement_units}
- Assay context: {row.assay_context}
- Notes: {row.notes}

## Intended use
Development and reliability evaluation of endpoint-specific early toxicity-screening models.

## Known limitations
Do not infer independence, assay comparability, or legal redistribution rights from the existence of this ML-ready table.
"""
    (card_dir / f"{row.dataset_name}.md").write_text(text, encoding="utf-8")

In [3]:
tox_spec = CONFIGS["data_config"]["sources"]["tox21"]
tox_path = ROOT / CONFIGS["data_config"]["raw_dir"] / "tox21" / tox_spec["raw_filename"]
tox = read_csv_preserve_missing(tox_path)
missingness = pd.DataFrame({
    "endpoint": tox_spec["label_columns"],
    "records": [len(tox)] * len(tox_spec["label_columns"]),
    "observed_labels": [int(tox[c].notna().sum()) for c in tox_spec["label_columns"]],
    "missing_labels": [int(tox[c].isna().sum()) for c in tox_spec["label_columns"]],
    "missing_rate": [float(tox[c].isna().mean()) for c in tox_spec["label_columns"]],
    "positive_prevalence_observed": [float(pd.to_numeric(tox[c], errors="coerce").dropna().mean()) for c in tox_spec["label_columns"]],
})
missingness.to_csv(ROOT / "data/metadata/tox21_missingness.csv", index=False)
assert (missingness["missing_labels"] >= 0).all()
atomic_write_json({"registry_rows": len(registry), "dataset_cards": len(list(card_dir.glob('*.md'))), "tox21_endpoints": len(missingness)}, ROOT / "reports/provenance_validation.json")
display(missingness)

,endpoint,records,observed_labels,missing_labels,missing_rate,positive_prevalence_observed
0,SR-p53,7831,6774,1057,0.134976,0.062445
1,SR-ATAD5,7831,7072,759,0.096922,0.037330
2,SR-ARE,7831,5832,1999,0.255268,0.161523
3,SR-MMP,7831,5810,2021,0.258077,0.158003


### Completion gate
Confirm that the declared artifacts exist before continuing to `04_molecular_standardization.ipynb`.